# Practice 57 — pandas + NumPy

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns

---

## Level 1 — Employee training scores

Quick reminder: `pd.wide_to_long(df, stubnames, i, j, sep='_', suffix=r'\w+')` folds columns with shared prefixes into rows. `stubnames` are the prefixes, `i` identifies each entity, `j` names the new suffix column.

Given `training` below, where each row is one employee and columns hold pre/post readings for `score` and `hours`:

- Reshape into long format. Each row should represent one `(employee, round)` pair, with `score` and `hours` as separate columns.
- Which department has the highest mean score across both rounds?
- Which round (`pre` or `post`) has the higher mean training hours?
- Use `np.corrcoef` to measure the correlation between `score` and `hours` across all `(employee, round)` pairs. Do employees who train longer score higher?

In [15]:
training = pd.DataFrame({
    'employee':   ['Ana', 'Ben', 'Cara', 'Dan', 'Eve', 'Fay'],
    'dept':       ['Sales', 'Eng', 'Sales', 'HR', 'Eng', 'HR'],
    'score_pre':  [72, 85, 68, 90, 78, 65],
    'score_post': [81, 91, 75, 93, 84, 80],
    'hours_pre':  [4.0, 3.5, 5.0, 2.5, 4.5, 6.0],
    'hours_post': [6.0, 5.5, 7.0, 4.5, 6.5, 8.0],
})

# Your code here

training_long = pd.wide_to_long(
    training, 
    stubnames= ['score','hours'],
    i = ['employee','dept'],
    j = 'round',
    sep = '_',
    suffix= r'\w+'
)

print('highest dept: ',training_long.groupby('dept')['score'].mean().idxmax())

print('highest round: ', training_long.groupby('round')['hours'].mean().idxmax())

np.corrcoef(training_long['score'], training_long['hours'])[0,1]
## the higher score the shorter training hours.

highest dept:  Eng
highest round:  post


np.float64(-0.2781113945413006)

---

## Level 2 — Penguins

Answer these questions about the seaborn `penguins` dataset. No function hints.

1. Build a crosstab of `island` vs `species`, normalized by row. On which island is the proportion of Chinstrap penguins highest?
2. Filter to Biscoe island only. Drop rows where `sex` is NaN. Build a crosstab of `sex` vs `species`, normalized by column — what fraction of each species on Biscoe is female?
3. Use `np.percentile` to find the 25th and 75th percentile of `body_mass_g` for each species separately.
4. Drop rows with any null in `bill_length_mm` or `flipper_length_mm`. Use `np.corrcoef` to check their correlation — are longer bills associated with longer flippers?

In [41]:
penguins = sns.load_dataset('penguins')

# Your code here

c = pd.crosstab(penguins['island'], penguins['species'], normalize='index')
print('on ',c['Chinstrap'].idxmax(), ' is the proportion of Chinstrap highest')

bp = penguins.loc[penguins['island'] == 'Biscoe'].dropna(subset = 'sex')

bc = pd.crosstab(bp['sex'],bp['species'], normalize='columns')
print(bc.loc['Female'])

print(penguins.groupby('species')['body_mass_g'].apply(lambda x: np.percentile(x.dropna(), [25,75])))

d = penguins.dropna(subset=['bill_length_mm', 'flipper_length_mm'])
cc = np.corrcoef(d['bill_length_mm'],d['flipper_length_mm'])[0,1]
print(f'correlation is {cc:.2f}')
print('yes')

on  Dream  is the proportion of Chinstrap highest
species
Adelie    0.500000
Gentoo    0.487395
Name: Female, dtype: float64
species
Adelie       [3350.0, 4000.0]
Chinstrap    [3487.5, 3950.0]
Gentoo       [4700.0, 5500.0]
Name: body_mass_g, dtype: object
correlation is 0.66
yes


---

## Level 3 — Titanic

Use the seaborn `titanic` dataset. No hints — choose the right tools.

1. Make `pclass` an ordered category: `3 < 2 < 1`. Filter to first- and second-class passengers using the category ordering. Among these, what fraction survived in each class?
2. Find the median `fare` using `np.percentile`. Among passengers whose fare is above the median, build a crosstab of `pclass` vs `survived`, normalized by row.
3. Drop rows where `age` is null. Is survival more strongly correlated with `fare` or with `age`? Report both `np.corrcoef` values and name the winner.

In [54]:
titanic = sns.load_dataset('titanic')

# Your code here

po = pd.CategoricalDtype([3,2,1], ordered=True)
titanic['pclass'] = titanic['pclass'].astype(po)

fs = titanic.loc[titanic['pclass']>3]
print(fs.groupby('pclass', observed=True)['survived'].mean())


mf = np.percentile(titanic['fare'],50)

print('median fare is: ', mf)

mt = titanic.loc[titanic['fare']>mf]
pd.crosstab(mt['pclass'], mt['survived'], normalize='index')


att = titanic.dropna(subset = 'age')

fsc = np.corrcoef(att['fare'], att['survived'])[0,1]
asc = np.corrcoef(att['age'], att['survived'])[0,1]

print('cor between fare and survival is: ', fsc)
print('cor between age and survival is: ', asc)

print('more correlated with fare')



pclass
2    0.472826
1    0.629630
Name: survived, dtype: float64
median fare is:  14.4542
cor between fare and survival is:  0.2681886168744788
cor between age and survival is:  -0.07722109457217768
more correlated with fare
